# L12a: How Do Micro Decisions Lead to Emergent Macro Trends?
In this lecture, we study agent-based models, in which we write down simple rules for how individual traders decide and then simulate many of them together to see what the market as a whole does. We work through two classical models: the thermal minority game, where we borrow the idea of temperature from physics to describe how agents choose among their strategies, and the Farmer–Joshi market, where value investors and trend followers trade through a market maker.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Identify the parts of an agent-based model:__ Describe the five parts that agent-based models share, the agents, the information they see, their decision rule, the aggregate, and the feedback from the aggregate to the next decision, and identify each part in the minority game and the Farmer–Joshi market.
> * __Derive the Boltzmann choice rule from maximum entropy:__ Derive the probabilities with which an agent chooses among its strategies by maximizing the entropy of the choice for a fixed expected score, and explain what happens at low and at high temperature.
> * __Connect agent rules to market behavior:__ Explain how the volatility of the minority game depends on the number of histories per agent and on the temperature, and how value investors and trend followers together produce fat tails and volatility clustering in the Farmer–Joshi market.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
> * An agent-based model has five parts: agents, the information each agent sees, a decision rule, an aggregate of the agents' decisions (the attendance, the net order, the price), and feedback from the aggregate to the next decisions.
> * Maximizing the entropy of the choice probabilities for a fixed expected score gives the Boltzmann rule $\pi_{s}=\exp(U_{s}/T)/\mathcal{Z}$. At $T\to 0$ the agent always plays its best strategy; as $T\to\infty$ it chooses uniformly at random.
> * Market-wide behavior comes from the rules, not from assumptions about prices. In the minority game, the volatility per agent depends on the number of histories per agent, $\alpha$, and on the temperature. In the Farmer–Joshi market, value investors and trend followers together produce fat tails and volatility clustering.
>
> __For understanding__ (you will not need to reproduce or memorize these):
> * The critical value $\alpha_{c}\approx 0.3374$, which we cite but do not derive, and the relation between the temperature and the change of the entropy with the expected score.
> * The threshold rules of the Farmer–Joshi traders and the parameter values of our simulation.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L12a) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples
__We will work through these examples in lecture:__

> [▶ Simulate the thermal minority game](CHEME-5660-L12a-Example-ThermalMinorityGame-Fall-2026.ipynb). In this example, we simulate a minority game in which agents choose to buy or sell each round with one of two fixed strategies, and we measure the volatility of the attendance per agent. We map how the volatility depends on the number of histories per agent, find its minimum near the critical value, and measure how the temperature of the strategy choice changes the volatility in the crowded phase.

> [▶ Simulate the Farmer–Joshi market](CHEME-5660-L12a-Example-FarmerJoshi-Fall-2026.ipynb). In this example, we simulate markets with value investors only, trend followers only, and both groups, and measure the excess kurtosis and the autocorrelations of the log returns, their absolute values, and the volume. We compare these statistics with the daily growth rates of a real ticker and vary the number of trend followers to see which stylized facts the model can match at the same time.
___

## In the News: The November 5 Sell-Off
On Thursday, November 5, 2026, U.S. stocks had their steepest one-day declines since the April 2025 tariff shock. The Nasdaq Composite fell 5.5%, into correction territory, and the S&P 500 fell about 4.4%, as investors worried that spending on AI data centers could slow after the midterm election results. The seven largest technology companies lost more than 1 trillion USD of market value ([CNBC](https://www.cnbc.com/2026/11/05/stock-market-selloff-ai-fears.html)).

> __Who did not buy the dip?__
>
> Retail investors usually buy when prices fall. "On a typical sell-off, retail buy the dip," Vanda Research wrote; "Today, that bid has effectively disappeared from individual names." Instead, retail investors sold a net 338 million USD of individual stocks, the largest one-day outflow since March 2020 ([CNBC](https://www.cnbc.com/2026/11/05/retail-traders-dump-the-most-stocks-in-single-day-since-covid-sell-off.html)).

__Connection to today's lecture:__ Investors who buy after prices fall act like the value investors of the Farmer–Joshi market, whose orders push the price back toward the value. Traders who sell after a fall act like its trend followers. Keep a question in mind as we build the model: how does the size of price moves change with the balance between the two groups?
___

## The Structure of an Agent-Based Model
Most of the models in this course describe prices directly: geometric Brownian motion, for example, assumes a distribution for the growth rate and simulates prices from it. An __agent-based model__ works from the other end. We describe individual decision makers, the __agents__, and the rules they follow, simulate many of them interacting, and then measure the prices or other market-wide quantities that come out. The question is which market-wide patterns follow from which individual rules.

Every agent-based model in this lecture has the same five parts:
* __Agents:__ the individual decision makers, often with different parameters.
* __Information:__ what each agent sees before it decides, such as recent outcomes or prices.
* __Decision rule:__ how each agent turns its information into an action.
* __Aggregate:__ the market-wide quantity that the actions add up to.
* __Feedback:__ how the aggregate changes what the agents see or prefer at the next step.

The feedback is what makes these models interesting: each agent reacts to an aggregate that all the agents create together. The table below maps the five parts onto the two models of this lecture.

| Part | Minority game | Farmer–Joshi market |
|---|---|---|
| Agents | $N$ agents, each holding $S$ fixed strategies | Value investors and trend followers, each with its own thresholds |
| Information | The last $m$ minority actions, one of $P=2^{m}$ histories $\mu(t)$ | The mispricing (value investors) or the price change over a lag (trend followers) |
| Decision rule | Choose a strategy by its score (greedy or Boltzmann), then play its action | Enter or exit a position when the signal crosses a threshold |
| Aggregate | The attendance $A(t)$, the sum of the actions | The net order, which the market maker turns into a price change |
| Feedback | The attendance updates every strategy's score and the history | The new price changes every trader's signal |
___

## The Minority Game
The minority game was introduced by [Challet and Zhang (1997)](https://doi.org/10.1016/S0378-4371(97)00419-6) as a simple model of a market in which it pays to be on the less crowded side: when most traders buy, the price rises, and the sellers get the better deal. An odd number $N$ of agents each choose, every round $t$, to buy ($+1$) or sell ($-1$). The agents on the minority side win.

__Information and strategies:__ All agents see the same information $\mu(t)$, the last $m$ minority actions, which is one of $P=2^{m}$ possible histories. Agent $i$ holds $S$ strategies, drawn at random when the game starts and never changed. A strategy is a table that assigns an action $a_{is}^{\mu}\in\{-1,+1\}$ to each history $\mu$; agent $i$ plays the action of the strategy it chooses, $a_{i}(t)$.

__Attendance:__ The attendance is the sum of the actions, and the minority side is $-\text{sign}\,A(t)$:

$$
A(t) = \sum_{i=1}^{N}a_{i}(t)
$$

Because $N$ is odd, $A(t)$ is never zero.

__Scores:__ After every round, each strategy gets a new score, whether or not its agent played it:

$$
U_{is}(t+1) = U_{is}(t) - \frac{a_{is}^{\mu(t)}A(t)}{P}
$$

A strategy that would have put its agent in the minority has an action with the opposite sign of $A(t)$, so its score rises; a strategy that would have put its agent in the majority loses score. Dividing by $P$ is the scaling of [Challet, Marsili, and Zecchina (2000)](https://doi.org/10.1103/PhysRevLett.84.1824). The minority action then becomes the newest entry of the history for the next round.

__Volatility:__ We measure how far the attendance moves from an even split with the volatility per agent:

$$
\frac{\sigma^{2}}{N} = \frac{\langle A^{2}\rangle}{N}
$$

where $\langle\cdot\rangle$ is the average over the rounds of a game. If every agent flipped a fair coin, $A(t)$ would have mean zero and variance $N$, so we would expect $\sigma^{2}/N=1$. Values above 1 mean that more agents end up on the same side than with coin flips, so the winners are a smaller minority; values below 1 mean that the agents split more evenly than coin flips.

The volatility depends on the number of histories per agent:

$$
\alpha = \frac{P}{N}
$$

When $\alpha$ is small, there are many agents for each history, and many of them switch to the same action at the same time. When $\alpha$ is large, each strategy is a long table, and the agents' choices barely affect each other's scores.

Let's put one round together. Panel A follows the information $\mu(t)$ to every agent, the agents' actions to the attendance $A(t)$, and the attendance back into the history and the scores. Panel B shows the two strategy tables of one agent and how the round changes their scores. The probabilities $\pi_{is}$ with which an agent chooses its strategy come from the Boltzmann rule, which we derive next.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L12a-MinorityGame.svg" width="1000" alt="Two panels. Panel A, one round: a row of past minority actions, with the last three (minus one, plus one, plus one) boxed and labeled history mu(t), and an empty dashed slot to their right. An arrow carries the history to agent 1, agent 2, and agent N, each a box noting that the agent draws its strategy s with probability pi_is. Below the agents are their actions, minus one, plus one, and minus one, labeled a_1(t), a_2(t), and a_N(t). The actions meet in a box, A(t) = sum of a_i(t) < 0, which leads to an orange square labeled minority, plus one. An orange arrow carries the minority action up the right side into the empty slot, labeled append to the history; a dashed blue arrow carries the attendance up the left side back to the agents, labeled update every score. Panel B, agent i's two strategies for m = 3: a table with one row for each of the eight histories and one column for each strategy, every entry plus one or minus one. The row for the current history, minus one, plus one, plus one, is shaded orange and marked mu(t); in it, strategy 1 plays minus one and strategy 2 plays plus one. Three rows below the table give the scores, 2.4 and 0.9; the Boltzmann probabilities at T = 1.5, 0.73 and 0.27; and the change after the round: strategy 1's score falls and strategy 2's rises, because the minority was plus one."/>
    </center>
</div>

### Choosing a strategy: from greedy to thermal
In the standard minority game, each agent plays the strategy with the highest score. [Cavagna, Garrahan, Giardina, and Sherrington (1999)](https://doi.org/10.1103/PhysRevLett.83.4429) let agents choose at random instead, with probabilities that favor high scores, and asked how strongly they should favor them. Let's derive that rule from a principle borrowed from physics.

#### Entropy of a choice
Drop the agent index $i$ for now. An agent chooses strategy $s$ with probability $\pi_{s}$, for $s=1,\dots,S$. The __entropy__ of the choice measures how spread out these probabilities are:

$$
\mathcal{H}(\pi) = -\sum_{s=1}^{S}\pi_{s}\ln\pi_{s}
$$

The entropy is zero when the agent picks one strategy for certain, and it is largest, equal to $\ln S$, when every strategy has probability $1/S$. A choice with high entropy is hard to predict; a choice with low entropy is nearly certain.

#### Maximizing entropy for a fixed expected score
Suppose we require the agent's choice to reach some expected score $\bar{U}$, but otherwise assume nothing about how it chooses. The choice that commits to the least, beyond what we require, is the one with the largest entropy. So we solve:

$$
\begin{align*}
\max_{\pi}\quad & -\sum_{s=1}^{S}\pi_{s}\ln\pi_{s}\\
\text{subject to}\quad & \sum_{s=1}^{S}\pi_{s} = 1\\
& \sum_{s=1}^{S}\pi_{s}U_{s} = \bar{U}
\end{align*}
$$

The first constraint makes the $\pi_{s}$ a probability distribution, and the second fixes the expected score. We take the scores not all equal and $\bar{U}$ strictly between their average and the largest score, so that the agent favors high scores and every $\pi_{s}$ is positive.

To solve it, we use __Lagrange multipliers__: we attach a multiplier to each constraint and look for the point where the derivative of the combined function, the Lagrangian, is zero. At the solution, each multiplier tells us how much the maximum entropy changes when we relax its constraint. With multipliers $\phi_{0}$ for the normalization and $\phi_{1}$ for the expected score, the Lagrangian is:

$$
\mathcal{L} = -\sum_{s=1}^{S}\pi_{s}\ln\pi_{s} - \phi_{0}\left(\sum_{s=1}^{S}\pi_{s}-1\right) + \phi_{1}\left(\sum_{s=1}^{S}\pi_{s}U_{s}-\bar{U}\right)
$$

Setting the derivative with respect to each $\pi_{s}$ to zero gives:

$$
\frac{\partial\mathcal{L}}{\partial\pi_{s}} = -\ln\pi_{s} - 1 - \phi_{0} + \phi_{1}U_{s} = 0
\quad\Longrightarrow\quad
\pi_{s} = e^{-1-\phi_{0}}\,e^{\phi_{1}U_{s}}
$$

The factor $e^{-1-\phi_{0}}$ is the same for every strategy, so the normalization constraint sets it: summing over $s$, $e^{-1-\phi_{0}}\sum_{k}e^{\phi_{1}U_{k}}=1$. The second derivative, $-1/\pi_{s}$, is negative, so this point is a maximum.

#### The Boltzmann rule and the temperature
We define the __temperature__ as $T=1/\phi_{1}$. The probability of choosing strategy $s$ is then the Boltzmann rule:

$$
\boxed{\pi_{s} = \frac{1}{\mathcal{Z}}\exp\left(\frac{U_{s}}{T}\right)}
\qquad\text{where}\qquad
\mathcal{Z} = \sum_{k=1}^{S}\exp\left(\frac{U_{k}}{T}\right)
$$

The normalizing sum $\mathcal{Z}$ is called the __partition function__, the name it has in statistical physics, where the same rule gives the probability of each state of a physical system, with the score replaced by the negative of the state's energy. The temperature has the units of the scores and controls how much the choice depends on them:
* __Low temperature, $T\to 0$:__ the strategy with the highest score has a probability close to 1, and in the limit the agent always plays it (ties share the probability equally). This is the standard, greedy minority game. When one strategy has the highest score, the entropy of the choice goes to zero.
* __High temperature, $T\to\infty$:__ every $U_{s}/T$ goes to zero, so every strategy has probability $1/S$ whatever the scores, and the entropy reaches its largest value, $\ln S$.
* __In between:__ only the differences between scores, compared with $T$, matter. With two strategies whose scores differ by $\Delta$, the better one is played with probability $e^{\Delta/T}/(e^{\Delta/T}+1)$, about 0.73 when $T=\Delta$.

The multiplier $\phi_{1}$ also has a meaning of its own. Raising the required expected score $\bar{U}$ forces the choice to concentrate on the best strategies, which lowers the maximum entropy, and the rate is $-d\mathcal{H}/d\bar{U}=\phi_{1}=1/T$. In thermodynamics, the temperature is defined the same way, through the change of entropy with energy. A low temperature means that a small gain in expected score costs a lot of entropy.

In the thermal minority game, every agent uses this rule each round, with its own current scores $U_{is}(t)$, so it plays strategy $s$ with probability $\pi_{is}$.

### The thermal minority game algorithm
Putting the pieces together, one game of the thermal minority game runs as follows:

__Initialize__: Given the number of agents $N$ (odd), the memory $m$, the number of strategies per agent $S$, the temperature $T\geq 0$, and the number of rounds $R$. Set $P=2^{m}$. For each agent $i$ and strategy $s$, draw the action $a_{is}^{\mu}\in\{-1,+1\}$ for every history $\mu$ at random, and set every score $U_{is}(1)\gets 0$. Draw the starting history $\mu(1)$ at random.

For each round $t=1,2,\dots,R$ __do__:
1. __Choose:__ For each agent $i$, compute $\pi_{is}(t)\gets\exp\left(U_{is}(t)/T\right)/\mathcal{Z}_{i}(t)$ for $s=1,\dots,S$, where $\mathcal{Z}_{i}(t)=\sum_{k=1}^{S}\exp\left(U_{ik}(t)/T\right)$, and draw the agent's strategy $s_{i}(t)$ with these probabilities. If $T=0$, __then__ choose the strategy with the highest score, breaking ties at random; if $T=\infty$, __then__ choose each strategy with probability $1/S$.
2. __Act:__ Each agent plays its strategy's action for the current history: $a_{i}(t)\gets a_{i\,s_{i}(t)}^{\mu(t)}$.
3. __Aggregate:__ Compute the attendance $A(t)\gets\sum_{i=1}^{N}a_{i}(t)$. The minority action is $-\text{sign}\,A(t)$.
4. __Score:__ Update every strategy of every agent, played or not: $U_{is}(t+1)\gets U_{is}(t)-a_{is}^{\mu(t)}A(t)/P$.
5. __Update the history:__ Drop the oldest of the $m$ minority actions in $\mu(t)$ and append $-\text{sign}\,A(t)$ to get $\mu(t+1)$.

__Output__: The attendance $A(1),\dots,A(R)$ and the volatility per agent $\sigma^{2}/N=\langle A^{2}\rangle/N$, averaged over the rounds after a burn-in.

### What the minority game shows
In the thermal minority game example, we simulated games with memory $m=6$ ($P=64$ histories) and $S=2$ strategies per agent, three games at each setting. Panel A plots the volatility per agent against $\alpha$ at $T=0$, and panel B plots it against the temperature for 641 agents ($\alpha\approx 0.1$):

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L12a-MinorityGame-Results.svg" width="1000" alt="Two panels from the thermal minority game example, memory m = 6 (P = 64 histories), two strategies per agent, three games at each setting. Panel A, the phase diagram at T = 0: the volatility per agent sigma^2/N against the histories per agent alpha = P/N, both on log axes, with one gray point per game and a red line through the averages. The averages fall from about 5 at alpha 0.06 through the dashed coin-flip line at 1 near alpha 0.2, to a minimum of 0.14 at alpha about 0.35, marked with an open circle just right of the dotted blue line at the critical value alpha_c = 0.3374, then rise to about 0.5 at alpha 3. The games spread out most just below alpha_c. Panel B, the volatility per agent against the temperature T on a log axis from 0.01 to 100,000 at alpha about 0.1: the averages stay near the dashed blue greedy-game line at 2.72 up to T about 0.3, drop steeply between T = 1 and 10, reach their lowest value, 0.46, near T = 100, below both the orange random-choice line at 1.08 and the dotted coin-flip line at 1, and rise slowly back toward the random-choice line at the highest temperatures."/>
    </center>
</div>

> __What do we see?__
>
> * __The phase diagram (A):__ In the crowded phase, small $\alpha$, the volatility per agent is above the coin-flip line. It falls to a minimum of 0.14 at $\alpha\approx 0.35$, close to the critical value $\alpha_{c}$ of Challet, Marsili, and Zecchina, and rises toward 1 in the dilute phase. At $\alpha_{c}$ the game goes through a __phase transition__, a sharp change in collective behavior when a parameter crosses a critical value.
> * __The temperature (B):__ Low temperatures give the greedy game (2.72), and very high ones approach random choice (1.08). In between, at $T=100$, the volatility per agent is 0.46, below both limits, as Cavagna and coauthors found. Their scores are divided by $N$ and ours by $P$, so their temperature is $\alpha$ times ours.

Let's look at the example:

> __Example:__
>
> [▶ Simulate the thermal minority game](CHEME-5660-L12a-Example-ThermalMinorityGame-Fall-2026.ipynb). In this example, we simulate one game and plot its attendance, map how the volatility per agent depends on the number of histories per agent, and measure how the temperature changes the volatility in the crowded phase.

Next, we add a price to an agent-based market.
___

## The Farmer–Joshi Market
The minority game has buyers and sellers but no price. [Farmer and Joshi (2002)](https://arxiv.org/abs/cond-mat/0012419) built an agent-based market with a price, two common trading strategies, and a market maker who sets the price from the traders' net order.

Let $p_{t}$ be the log price and $v_{t}$ the log value of the asset at step $t$. Trader $i$ holds a position $x_{t}^{(i)}$ of $-c_{i}$ (short), $0$, or $+c_{i}$ (long), and its order is the change in its position, $\omega_{t+1}^{(i)}=x_{t+1}^{(i)}-x_{t}^{(i)}$. The market maker moves the log price in proportion to the net order:

$$
p_{t+1} = p_{t} + \frac{1}{\lambda}\sum_{i}\omega_{t+1}^{(i)} + \xi_{t+1}
$$

where $\lambda$ is the __liquidity__, the net order that moves the log price by one, and $\xi_{t+1}$ is a Normal noise term with standard deviation $\sigma_{\xi}$ that stands for everything the model leaves out. The log return is $r_{t+1}=p_{t+1}-p_{t}$.

Both kinds of traders use the same __threshold rule__ on different signals. A trader enters a position when its signal crosses an entry threshold $\pm\chi^{\text{in}}_{i}$ and exits when the signal comes back across an exit threshold $\pm\chi^{\text{out}}_{i}$, with $\chi^{\text{in}}_{i}>\chi^{\text{out}}_{i}$. The position size is $c_{i}=a\left(\chi^{\text{in}}_{i}-\chi^{\text{out}}_{i}\right)$, so traders with a wider gap between their two thresholds trade larger positions.
* __Value investors__ watch the mispricing $p_{t}-\left(v_{t}+v^{(i)}\right)$, where the offset $v^{(i)}$ makes each investor's estimate of the value different. They sell when the asset is overpriced by more than $\chi^{\text{in}}_{i}$ and buy when it is underpriced by more than $\chi^{\text{in}}_{i}$, so their trading pushes the price toward the value.
* __Trend followers__ watch the price change over their own lag $\theta_{i}$, $p_{t}-p_{t-\theta_{i}}$. They buy when the price has risen by more than $\chi^{\text{in}}_{i}$ and sell when it has fallen by more than $\chi^{\text{in}}_{i}$, so their trading pushes the price further in the direction it has been moving.

The figure below shows one step of the market (panel A) and the threshold rule of a value investor as three states, long, flat, and short (panel B). We write $\zeta^{(i)}_{t}$ for the signal that trader $i$ watches.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L12a-FarmerJoshi.svg" width="1000" alt="Two panels. Panel A, one step of the Farmer–Joshi market: the log value v_t feeds a red box of value investors, whose signal is zeta_t^(i) = p_t minus (v_t plus v^(i)); next to it, a blue box of trend followers, whose signal is zeta_t^(i) = p_t minus p_(t minus theta_i). Arrows labeled orders omega_(t+1)^(i) from both boxes meet in a summation node; the net order goes to the market maker, p_(t+1) = p_t + (1/lambda) times the sum of the orders + xi_(t+1), with the noise xi_(t+1) entering from the right. The market maker produces the new log price p_(t+1), and orange arrows carry it back up both sides to the two groups, labeled new price, new signals. Panel B, the threshold rule of value investor i as three states, long plus c_i on the left, flat 0 in the middle, and short minus c_i on the right. From flat, the investor sells (goes short) when zeta exceeds chi_in (overpriced) and buys (goes long) when zeta falls below minus chi_in (underpriced). From short, it returns to flat when zeta falls below chi_out; from long, when zeta rises above minus chi_out. Below: otherwise the position stays the same; the position size is c_i = a (chi_in minus chi_out); trend followers use the same rule with the signs reversed, buying when zeta exceeds chi_in and selling when zeta falls below minus chi_in."/>
    </center>
</div>

### The Farmer–Joshi algorithm
One run of the market follows these steps:

__Initialize__: Given the value investors and the trend followers, each with an entry threshold $\chi^{\text{in}}_{i}$ and an exit threshold $\chi^{\text{out}}_{i}<\chi^{\text{in}}_{i}$; the value investors' offsets $v^{(i)}$; the trend followers' lags $\theta_{i}$; the scale factor $a$, the liquidity $\lambda$, the noise standard deviations $\sigma_{\xi}$ and $\sigma_{\eta}$, and the number of steps $R$. Set each position size $c_{i}\gets a\left(\chi^{\text{in}}_{i}-\chi^{\text{out}}_{i}\right)$, every position $x^{(i)}_{0}\gets 0$, and the starting log price $p_{0}$ and log value $v_{0}$.

For each step $t=0,1,\dots,R-1$ __do__:
1. __Signals:__ For each value investor, $\zeta^{(i)}_{t}\gets p_{t}-\left(v_{t}+v^{(i)}\right)$. For each trend follower with $t\geq\theta_{i}$, $\zeta^{(i)}_{t}\gets p_{t}-p_{t-\theta_{i}}$; a trend follower with $t<\theta_{i}$ has no signal yet and keeps $x^{(i)}_{t+1}\gets 0$.
2. __Positions:__ For each value investor:
    - If $x^{(i)}_{t}=0$ and $\zeta^{(i)}_{t}>\chi^{\text{in}}_{i}$, __then__ sell: $x^{(i)}_{t+1}\gets -c_{i}$. If $x^{(i)}_{t}=0$ and $\zeta^{(i)}_{t}<-\chi^{\text{in}}_{i}$, __then__ buy: $x^{(i)}_{t+1}\gets +c_{i}$.
    - If $x^{(i)}_{t}=-c_{i}$ and $\zeta^{(i)}_{t}<\chi^{\text{out}}_{i}$, __then__ exit: $x^{(i)}_{t+1}\gets 0$.
    - If $x^{(i)}_{t}=+c_{i}$ and $\zeta^{(i)}_{t}>-\chi^{\text{out}}_{i}$, __then__ exit: $x^{(i)}_{t+1}\gets 0$.
    - Otherwise, $x^{(i)}_{t+1}\gets x^{(i)}_{t}$.

    Trend followers use the same rule with the signs of the positions reversed: they buy when $\zeta^{(i)}_{t}>\chi^{\text{in}}_{i}$ and sell when $\zeta^{(i)}_{t}<-\chi^{\text{in}}_{i}$.
3. __Orders:__ For every trader, $\omega^{(i)}_{t+1}\gets x^{(i)}_{t+1}-x^{(i)}_{t}$.
4. __Price:__ Draw $\xi_{t+1}$ from a Normal distribution with mean zero and standard deviation $\sigma_{\xi}$, and set $p_{t+1}\gets p_{t}+\frac{1}{\lambda}\sum_{i}\omega^{(i)}_{t+1}+\xi_{t+1}$.
5. __Value:__ Draw $\eta_{t+1}$ from a Normal distribution with mean zero and standard deviation $\sigma_{\eta}$, and set $v_{t+1}\gets v_{t}+\eta_{t+1}$.

__Output__: The log prices $p_{0},\dots,p_{R}$, the log values $v_{0},\dots,v_{R}$, the log returns $r_{t+1}=p_{t+1}-p_{t}$, and the volume $\sum_{i}\lvert\omega^{(i)}_{t+1}\rvert$ at each step.

We draw each trader's thresholds, offset, and lag at random from the ranges in Farmer and Joshi's Table 1, where every entry threshold is positive and every exit threshold is zero or negative, with 1,200 traders of each kind. We make two changes, so our simulation is a variant of theirs, not a replication. Their Table 1 lists $\sigma_{\xi}=0.35$; with that value, the price in our simulations drifts much farther from the value than in the paper's figures of price and value, so we use $\sigma_{\xi}=0.05$. And instead of their series of S&P 500 dividends, our log value follows a random walk.

### What the Farmer–Joshi market shows
Previously, we measured stylized facts of daily growth rates: fat tails, little autocorrelation of the growth rates themselves, and volatility clustering, where large moves follow large moves. In the Farmer–Joshi example, we simulated each market for 20,000 steps. With value investors only, the lag-1 autocorrelation of the log returns was small and negative ($-0.024$); with trend followers only, it was 0.91. Panel A compares the market with both groups with `SPY` daily growth rates from 2014 to 2024, and panels B and C vary the number of trend followers alongside 1,200 value investors:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L12a-FarmerJoshi-Results.svg" width="1000" alt="Three panels from the Farmer–Joshi example (our variant, noise 0.05). Panel A, volatility clustering: the autocorrelation of the absolute log returns of the market with both groups (red) and of the absolute daily growth rates of SPY (black) at lags 1 to 50. The model starts at 0.67 at lag 1 and falls to about zero within five steps, then stays within about 0.05 of zero; SPY starts at 0.36 and stays between about 0.06 and 0.43 out to lag 50. Panel B, fat tails: the excess kurtosis of the log returns against the number of trend followers, 0 to 1,500, with 1,200 value investors, one gray point per market and a red line through the averages of three markets. It stays near zero up to 600 trend followers; at 900 one market jumps to about 100 while the other two stay near zero; at 1,200 and 1,500 every market is between about 27 and 60, above the dashed SPY line at 13.8. Panel C, return autocorrelation: the lag-1 autocorrelation of the log returns against the number of trend followers. It is near zero up to 600, and from 900 on the markets spread from about 0.05 to 0.5, all above the dashed SPY line at minus 0.11."/>
    </center>
</div>

> __What do we see?__
>
> * __Volatility clustering (A):__ Both series have positive autocorrelation of their absolute values, so large moves follow large moves. The model's clustering fades within a few steps, while that of `SPY` lasts for weeks.
> * __The mix of traders (B, C):__ With 600 or fewer trend followers, the excess kurtosis and the lag-1 autocorrelation stay near zero. With 1,200 or more, every market has fat tails, with an excess kurtosis above that of `SPY`, and a lag-1 autocorrelation above the small negative value of `SPY`. At every mix, at least one of the three statistics is far from that of `SPY`.

Let's look at the example:

> __Example:__
>
> [▶ Simulate the Farmer–Joshi market](CHEME-5660-L12a-Example-FarmerJoshi-Fall-2026.ipynb). In this example, we simulate markets with value investors only, trend followers only, and both groups, measure the stylized facts of the market with both groups, compare them with a real ticker, and vary the number of trend followers.

Two simple trading rules and a market maker produce fat tails and volatility clustering without either being built in, but not every stylized fact at once.
___

## Summary
This lecture introduced agent-based models, derived the Boltzmann rule for choosing among strategies from maximum entropy, and used the minority game and the Farmer–Joshi market to connect individual trading rules to market-wide behavior.

> __Key Takeaways:__
>
> * __Agent-based models build markets from individual rules:__ We described five parts that agent-based models share, the agents, their information, their decision rule, the aggregate, and the feedback from the aggregate, and identified them in the minority game and the Farmer–Joshi market. Simulating the rules, rather than assuming a distribution for prices, lets us ask which market-wide patterns follow from which behaviors.
> * __Maximum entropy gives the Boltzmann choice rule:__ We derived the probabilities with which an agent chooses among its strategies by maximizing the entropy of the choice for a fixed expected score, using Lagrange multipliers. The multiplier on the expected score is the inverse of the temperature, which moves the agent from always playing its best strategy at low temperature to choosing uniformly at random at high temperature.
> * __Simple rules produce market-wide patterns:__ We saw that the volatility of the minority game depends on the number of histories per agent, with a minimum near a critical value, and that a moderate temperature lowers the volatility in the crowded phase below both limits. In the Farmer–Joshi market, value investors and trend followers together produced fat tails and volatility clustering, although not every stylized fact of real growth rates at once.

These models show how the behavior of a whole market can follow from the rules of the traders in it.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___